In [2]:
import torch

torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 4060 Ti'

In [3]:
import os

# TODO: Check this
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

In [4]:
from huggingface_hub import notebook_login


notebook_login(new_session=False)

User is already logged in.


In [5]:
from datasets import load_dataset, DatasetDict, Audio

esther = DatasetDict()
esther["train"] = load_dataset("../data/44.1/chunks/694", split="train")
esther["test"] = load_dataset("../data/44.1/chunks/737", split="train")
print(esther)
print(esther["train"][0])
esther = esther.cast_column("audio", Audio(sampling_rate=16000))
print(esther["train"][0])

Resolving data files:   0%|          | 0/111 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/105 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio'],
        num_rows: 111
    })
    test: Dataset({
        features: ['audio'],
        num_rows: 105
    })
})
{'audio': {'path': '/workspaces/whisper/data/44.1/chunks/694/chunk_0.mp3', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00781925,
       -0.00936907, -0.00756717]), 'sampling_rate': 44100}}
{'audio': {'path': '/workspaces/whisper/data/44.1/chunks/694/chunk_0.mp3', 'array': array([-1.88653211e-14, -1.69484110e-13,  6.50063445e-13, ...,
       -7.68004823e-03, -8.08591582e-03, -8.97248276e-03]), 'sampling_rate': 16000}}


In [6]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor


feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")

In [7]:
def prepare_dataset(batch):
    # reformat columns
    audio = batch["audio"]  # samples at 16kHz

    # compute log mel spectrogram
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode transcription text to token ids
    # batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [8]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
import torch

from dataclasses import dataclass
from typing import Union, Dict, Any, List


@dataclass
class DataCollatorSpeech:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label features
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        label_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore the loss correctly
        labels = label_batch["input_ids"].masked_fill(label_batch.attention_mask.ne(1), -100)

        # if bos (beginging of string?) token is appended in previous tokenzation step,
        # remove it, as it is appended later anyway
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeech(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)